# Inference in Bayesian Networks

Now that we have a Bayesian network, what can we do with it? Well, since the Bayesian network is simply a representation of the full joint probability distribution we can do anything we would have done with the joint distribution. Any quantity that could be calculated from the joint distribution can be computed from the Bayesian network.

The straightforward way to do this would simply be to construct the full joint distribution directly, but that would be to miss the point. One of the reasons we use Bayesian networks is to reduce the number of parameters needed to represent the joint distribution and so to construct the full joint distribution from the network would be both wasteful and would also render the efficient and meaning representation provided by the network redundant.

## Exact Inference

Consider our example problem with the following DAG

 ![DAG for calling the Fire Brigade](./figures/fire_dag.png)

that represents the factorisation of the joint distribution as.

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

We have defined all of the probabilities in the terms in the factorisation and so we can compute the joint distribution. For convenience,

| $P(R)$ | $P(\lnot R)$ |
|:------:|:------------:|
| 0.3    | 0.7          |

</br></br>


| $P(C)$ | $P(\lnot C)$ |
|:------:|:------------:|
| 0.1    | 0.9          |

</br></br>



| $P(D)$ | $P(\lnot D)$ |
|:------:|:------------:|
| 0.5    | 0.5          |

</br></br>

|           | $P(A\vert C)$ | $P(\lnot A\vert C)$ |
|----------:|:-------------:|:-------------------:|
| $C$       | 0.01          |  0.99               |
| $\lnot C$ | 0.0           |  1.00               |

</br></br>


|           | $P(F\vert A)$ | $P(\lnot F\vert A)$ |
|----------:|:-------------:|:-------------------:|
| $A$       | 0.30          |  0.70               |
| $\lnot A$ | 0.001         |  0.990              |

</br></br>



|                  | $P(T\vert F,D)$ | $P(\lnot T\vert F,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $D,F$            | 0.99            |  0.01                 |
| $D,\lnot F$      | 0.60            |  0.40                 |
| $\lnot D,F$      | 0.70            |  0.30                 |
| $\lnot D,\lnot F$| 0.01            |  0.99                |

</br></br>


|                  | $P(S\vert F,D)$ | $P(\lnot S\vert F,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $T,F$            | 0.99            |  0.01                 |
| $T,\lnot F$      | 0.30            |  0.70                 |
| $\lnot D,F$      | 0.99            |  0.01                 |
| $\lnot D,\lnot F$| 0.001           |  0.999                |

**WARNING**: this is quite intricate and fiddly, but there are some well-define conventions we can use to make it simpler.

1. Use a sensible naming convention. We will use:
- $P(A)\mapsto$`PA`
- $P(A\vert B)\mapsto$`PA_B`
- $P(A\vert B,C)\mapsto$`PA_BC`

2. Order the variables consistently. We will choose that
- Index zero corresponds to an outcome being False, and index one to True
- The indices are ordered in the same way as in the variable name, such that `PA_BC` has elements `PA_BC[ia][ib][ic]`

We then have

```PA = np.array([P(A=False),P(A=True)])```

```PA_B = np.array([[P(A=False|B=False),P(A=False|B=True)],[P(A=True|B=False),P(A=True|B=True)]])```

With these rules in mind we can write down our distributions

In [1]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])


At this point, take a step back and check that the way we have done this is correct with a manual calculation.

We now have anough information to calculate $P(A)$.

Since $P(A,C) = P(A\vert C)P(C)$ and $P(A) = \sum_C P(A,C)$ we have $P(A)=\sum_C P(A\vert C)P(C)$.

Let's check that this works with a manual calculation:

$P(\lnot A) = \sum_C P(\lnot A\vert C)P(C) = P(\lnot A\vert C)P(C) + P(\lnot A\vert \lnot C)P(\lnot C) = 0.99\times 0.1 + 1.0*0.9 = 0.999$

$P( A) = \sum_C P( A\vert C)P(C) = P( A\vert C)P(C) + P( A\vert \lnot C)P(\lnot C) = 0.01\times 0.1 + 0.0*0.9 = 0.001$

In [2]:
PA = (PA_C*PC).sum(axis=1)
print(PA)

[0.999 0.001]


In fact, there is an even easier way to do this: this is exactly equivalent to a matrix multiplication:

In [3]:
PA = PA_C@PC
print(PA)

[0.999 0.001]


So we have a nice neat notation here. Let's add a few more terms:

In [4]:
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])


Pause again to check - we are now in a position to compute $P(T)$. Here we have to be careful with the order in which we multiple things. Let's compute one of these by hand just to make sure:

$P(T) = \sum_{CAF} P(T\vert F)P(F\vert A)P(A\vert C)P(C)$



In [5]:
# First compute P(F)
PF = PF_A@PA
print(PF)

# Then PT_F
PT_F = PT_FD@PD
print(PT_F)

# Finally PT
PT = PT_F @ PF
print(PT)

# Or all in one go:

PT = PT_FD @ PD @ PF_A @ PA
print(PT)

[0.998701 0.001299]
[[0.695 0.155]
 [0.305 0.845]]
[0.69429854 0.30570146]
[0.69429854 0.30570146]


Note that since we are working with matrices here, the order in which we compute things is crucial.

Is it right? The manual calculation is rather tedious so we just do a sanity check. This suggests that the cat is twice as likely to not be up the tree as it is to be up the tree. The chances of a house fire are very low, and so we should expect this to be dominated by presence/absence of the dog. We can use

$P(T) = \sum_D P(T\vert D)P(D) \approx \sum_D P(T\vert D,F=False)P(D)$

The distribution $P(T\vert D,F=False)$ is

|                  | $P(T\vert F=False,D)$ | $P(\lnot T\vert F=False,D)$ |
|-----------------:|:---------------:|:---------------------:|
| $D$      | 0.60            |  0.40                 |
| $\lnot D$| 0.01            |  0.99                 |


and so

$P(T) \approx P(T\vert\lnot D)P(\lnot D) + P(T\vert D)P(D) = 0.99\times 0.5 + 0.4\times 0.5) \approx 0.7$

and

$P(\lnot T) \approx P(T\vert\lnot D)P(\lnot D) + P(T\vert D)P(D) = 0.01\times 0.5 + 0.6\times 0.5) \approx 0.3$

Close enough to give us confidence that we have set this up right.

Finally, we can compute joint distributiuons from our conditional distributions. For example, $P(A,C)=P(A\vert C)P(C)$. How do we compute this? There is no summation required here so this is an element-wise multiplication:

In [6]:
print(PA_C)
print(PC)

PAC = PA_C*PC
print(PAC)

[[1.   0.99]
 [0.   0.01]]
[0.9 0.1]
[[0.9   0.099]
 [0.    0.001]]


This is normalised and so is a valid joint distribution. Is it correct?

$P(\lnot A,\lnot C) = P(\lnot A\vert \lnot C)P(\lnot C) = 1.0\times0.9 = 0.9$

$P(\lnot A, C) = P(\lnot A\vert C)P(C) = 0.99\times0.1 = 0.099$

$P(A,\lnot C) = P(A\vert \lnot C)P(\lnot C) = 0.0 \times0.099 = 0.0$

$P(A,C) = P(A\vert C)P(C) = 0.01\times0.1 = 0.001$

## Approximate Inference

Exact inference is conceptually straightforward, but practically difficult, error-prone, and very expensive, scaling very badly with the number of variables. It also **fails completely** for continuous variables where the calculations involved are essentially intractable. For these reasons it very rare to use this approach in practice. So what do we do instead? The dominant class of algorithm for inference on Bayesian networks is based on *sampling*.

In sampling approaches to inference, we use systematic random draws from the factors in our factorised joint distribution to build up an approximation of the distribution we want to compute (which is typically a conditional distribution that is not included in the factorisation). There are many ways in which sampling can be done, we will look at some of the most common.

### Direct Sampling

The most straightforward approach to sampling is a technique called the *direct sampling* method. This is the simplest and most "obvious" approach. It does not scale particularly well but is conceptually simple and very easy to implement so can be used quite effectively as long as a problem is not too large (ie many factors in the distribution, or many possible values for each factor).


Imagine we have a coin and we want to know its distribution of heads vs tails if we toss it. How can we do this? We cannot determine the distribution easily from first principles, although we typically make some assumptions about the symmetry of the coin to conclude that $P(H)=P(T)=0.5$. How could we verify that our assumptions were correct though?

There is a very simple answer to this. We *sample* the distribution by tossing the coin a large number of times ($N$) and counting how many times each of the two outcomes occurs ($N_H, N_T$). We then estimate the distribution as $P(H)=N_H/N$ and $P(T)=N_T/N$. If the number of heads is roughly equal to the number of tails then we have approximately verified that that coin is fair and that $P(H)=P(T)$. We can simulate a fair coin using the Binomial Distribution with $p=0.5$

In [7]:
import numpy as np
ntosses = 100
nheads = np.random.binomial(ntosses,0.5)
print(f"P(H)={nheads/ntosses}  P(T)={1-nheads/ntosses}")

P(H)=0.43  P(T)=0.5700000000000001


Let us now see how to apply this to a simple Bayesian Network.

![Simple Bayesian Network](./figures/dag.png)

(This networks represents the relationship between studying **G**eography or **M**athematics at high school and careers in **F**inance or **P**rogramming). The distributions are:

$P(G) = \begin{pmatrix}0.6\\0.4\end{pmatrix}$

$P(M) = \begin{pmatrix}0.75\\0.25\end{pmatrix}$

$P(F\vert M) = \begin{pmatrix}0.8 & 0.4\\ 0.2 & 0.6\end{pmatrix}$

$P(P\vert M) = \begin{pmatrix}0.65 & 0.45\\ 0.35 & 0.55\end{pmatrix}$

We could quite easily use direct computation to compute any distribution we could possibly want here, but we can also use *sampling* to compute. Indeed, when a problem becomes very large (many variable, many possibilities for each variable) then we will have to use sampling.

As we did with the simple example of the coin toss, we will draw samples and count outcomes to build up an approximation of the distribution. Here, this is to approxmate the joint distribution. However, we only have the factorisation to work with. How do we do this?

We can sequentially sample from the terms in the factorised distribution. Starting with the independent variable, we draw a sample. We then use this draw to condition subsequent distributions until we have taken a draw from all of the factors. Let's work through an example:

* Draw one sample from $P(G) = (0.6,0.4)$. Answer is (say) False.
* Draw one sample from $P(M) = (0.75,0.25)$. Answer is (say) True.
* Draw one sample from $P(F\vert M=True) = (0.4,0.6)$. Answer is True.
* Draw one sample from $P(P\vert M=True) = (0.45,0.55)$. Answer is False.

We have now drawn one sample from the joint distribution and obtained (False,True,True,False). Given a large number of such samples, we can compute anything we want. Let's try it out. We will code this up very explicitly which will be slightly inefficient but we want to maintain transparency whilst we are learning.

### Likelihood weighting

We have previously introduce the *discrete sampling* method and shown how it can be adapted for conditional distributions using *rejection sampling*. In this technique, we sample randomly from across the parameter space and then only retain those samples which have the correct values of the *evidence variables*: variables that are considered to be fixed by our accumulated knowledge, or by our desired quantity. This will be clearer with an example.

Let's say that we want to compute the probability that someone with a career in programming but not in the finance sector studies mathematics, $P(M\vert P=True, F=False)$. In rejection sampling we would allow the sampler to generate events with $P$ both True and False, and then reject all those samples where $P$ is False. Can we compute only those values that we would need to do this? Yes, with an approach called *likelihood reweighting* in which we fix certain variables to the desired values and then reweight the answers in a compensate for the fact that the variable don't have those values all of the time.

An alternative approach is to only generate samples that are consistent with the evidence, that is, that have the same value as the evidence variables. This means that no events are wasted which is good. However, we cannot just fix those values because this would bias the outcome. To see this, think about how to do this for $P(M\vert P=True, F=False)$. Here, $P$ and $F$ are so-called evidence variables (image we had "observed" them and found that they took these values: they are evidence). We could just set $P=True$, $F=False$, but what if $P(F=False)=0$? In this extreme case the conditions we are simulating could never occur. We need to compensate for this. In outline, we do this as follows:

* Fix all the evidence variables to their observed values
* Sample non-evidence variables conditioned on the fixed evidence variables
* Compute the probability that the sample you have just drawn would give rise to the evidence that you have fixed.
* Include your sample in the trace, weighted by this probability.

Let's illustrate this by working through our example.

* Fix $P=True$.
* Set weight $w=1.0$.
* $G$ is an evidence variable and is fixed at False. Set $w\leftarrow w\times P(G=False) = 0.6$.
* $M$ is not evidence. Sample from $P(M) = (0.75,0.25)$, suppose this returns True.
* Sample from $P(F\vert M=True) = (0.4,0.6)$, suppose this returns False.
* $P$ is an evidence variable and is True. Set $w\leftarrow w\times P(P=True\vert M=True) = 0.6\times 0.55 = 0.33$.
* Record $(G,M,F,P) = (False, True, False, True)$ with weight 0.33.
* Count samples as before, scaled by the weights assigned to each row of the trace.

Why does this work? Firstly, all of the samples generate have the observed values of the evidence variables and so all of the samples can be used. This give improved accuracy results or faster convergens for a given level of accuracy. Secondly, it can be shown rigourously that weighting of the likelihoods in this way is fully consistent with direct sampling from the desired posterior distribution. A proof can be found in Section 14.5.1 of Russell and Norvig (Fifth Edition). The main disadvantage is the flexibility: we have to fix the evidences variables, and if we want to compute for a different set of evidence then we have to regenerate a new trace.

Let's implement this. First, let's do the calculation:



In [8]:

import numpy as np

PG = np.array([0.6,0.4])
PM = np.array([0.75,0.25])
PF_M = np.array([[0.8,0.4],[0.2,0.6]])
PP_M = np.array([[0.65,0.45],[0.35,0.55]]) 

Let us first generate something we know. We'll compute $P(\lnot F\vert M))$ We need to set $M=1$ as evidence.

In [9]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Set M=1 as evidence and reweight
    M = 1
    w *= PM[M]
    # Draw one sample from P(F|M)
    F = np.random.binomial(1,PF_M[1,M])
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence (ie conditioned on M=1) by construction, so we do the following
# Find all rows in the trace where F=0
FEq0MEq1 = Trace[np.where(Trace[:,2]==0)]
# Sum the weights in the subset of the trace we want, and divide by the sum of the weights in the full trace
PFEq0MEq1 = (FEq0MEq1[:,4].sum()) / Trace[:,4].sum()
print(f"P(F=0|M=1) = {PFEq0MEq1} (expected {PF_M[0,1]})")


P(F=0|M=1) = 0.4041 (expected 0.4)


Check another one of the inputs.

In [10]:
print(f"P(F=1|M=1) = {Trace[np.where(Trace[:,2]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PF_M[1,1]})")
print(f"P(P=0|M=1) = {Trace[np.where(Trace[:,3]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PP_M[0,1]})")
print(f"P(P=1|M=1) = {Trace[np.where(Trace[:,3]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PP_M[1,1]})")


P(F=1|M=1) = 0.5959 (expected 0.6)
P(P=0|M=1) = 0.4579 (expected 0.45)
P(P=1|M=1) = 0.5421 (expected 0.55)


Now something that isn't an input: $P(M\vert F)$. We can compute exactly to check:

In [11]:
PF = PF_M @ PM
PM_F = (PF_M*PM).T/PF
print(f"P(M|F) = {PM_F}")

P(M|F) = [[0.85714286 0.5       ]
 [0.14285714 0.5       ]]


First fix $F=0$

In [12]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Sample M
    M  = np.random.binomial(1,PM[1])
    # Set evidence
    F = 0
    w *= PF_M[F,M]
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence by construction, so all we need to do is count the rows where $M=0$
print(f"P(M=0|F=0) = {Trace[np.where(Trace[:,1]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,0]})")
print(f"P(M=1|F=0) = {Trace[np.where(Trace[:,1]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,0]})")

P(M=0|F=0) = 0.8572734544623473 (expected 0.8571428571428572)
P(M=1|F=0) = 0.14272654553765282 (expected 0.8571428571428572)


Now for F=1

In [13]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Sample M
    M  = np.random.binomial(1,PM[1])
    # Set evidence
    F = 1
    w *= PF_M[F,M]
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence by construction, so all we need to do is count the rows where $M=0$
print(f"P(M=0|F=1) = {Trace[np.where(Trace[:,1]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,1]})")
print(f"P(M=1|F=1) = {Trace[np.where(Trace[:,1]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PM_F[0,1]})")

P(M=0|F=1) = 0.5008006405124099 (expected 0.5)
P(M=1|F=1) = 0.49919935948759003 (expected 0.5)


Finally, let's compute $P(M\vert \lnot F,P)$. This is actually rather tricky to compute by hand, so let's compute by direct sampling:

In [14]:
N = 10000
Trace = np.zeros([N,4])
for i in range(N):
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Draw one sample from P(M)
    M = np.random.binomial(1,PM[1])
    # Draw one sample from P(F|M)
    F = np.random.binomial(1,PF_M[1,M])
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P])

# Rows where F=0
x = Trace[np.where(Trace[:,2]==0)]
# Rows where F=0 and P=1
x = x[np.where(x[:,3]==1)]

PM_NotFP = x[np.where(x[:,1]==1)].shape[0]/x.shape[0]
PNotM_NotFP = x[np.where(x[:,1]==0)].shape[0]/x.shape[0]

print(f"P(M=0|F=0,P=1) = {PM_NotFP}")
print(f"P(M=0|F=0,P=1) = {PNotM_NotFP}")

P(M=0|F=0,P=1) = 0.20601503759398496
P(M=0|F=0,P=1) = 0.793984962406015


In [15]:
N = 10000
Trace = np.zeros([N,5])
for i in range(N):
    w = 1.0
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Sample M
    M  = np.random.binomial(1,PM[1])
    # Set evidence
    F = 0
    w *= PF_M[F,M]
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    P = 1
    w *= PP_M[P,M]
    Trace[i:] = np.array([G,M,F,P,w])

# Since all rows of the trace are consistent with the evidence by construction, so all we need to do is count the rows where $M=0$
print(f"P(M=0|F=0,P=1) = {Trace[np.where(Trace[:,1]==0)][:,4].sum()/Trace[:,4].sum()} (expected {PNotM_NotFP})")

print(f"P(M=1|F=0,P=1) = {Trace[np.where(Trace[:,1]==1)][:,4].sum()/Trace[:,4].sum()} (expected {PM_NotFP})")


P(M=0|F=0,P=1) = 0.7917509586642106 (expected 0.793984962406015)
P(M=1|F=0,P=1) = 0.20824904133578911 (expected 0.20601503759398496)


In [16]:
import numpy as np

PG = np.array([0.6,0.4])
PM = np.array([0.75,0.25])
PF_M = np.array([[0.8,0.4],[0.2,0.6]])
PP_M = np.array([[0.65,0.45],[0.35,0.55]])

In [17]:
N = 50000
Trace = np.zeros([N,4])
for i in range(N):
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Draw one sample from P(M)
    M = np.random.binomial(1,PM[1])
    # Draw one sample from P(F|M)
    F = np.random.binomial(1,PF_M[1,M])
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P])

print(Trace)

[[0. 1. 0. 1.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]
 ...
 [0. 0. 1. 0.]
 [1. 0. 0. 1.]
 [1. 1. 1. 0.]]


Let's do some basic stats on the Trace

In [18]:
PF = PF_M@PM
PP = PP_M@PM

print(f"P_G = {Trace[:,0].sum()/N} (expected {PG[1]})")
print(f"P_M = {Trace[:,1].sum()/N} (expected {PM[1]})")
print(f"P_F = {Trace[:,2].sum()/N} (expected {PF[1]})")
print(f"P_P = {Trace[:,3].sum()/N} (expected {PP[1]})")

P_G = 0.40086 (expected 0.4)
P_M = 0.25066 (expected 0.25)
P_F = 0.3015 (expected 0.30000000000000004)
P_P = 0.40014 (expected 0.39999999999999997)


Pretty good! Can we now estimate some other quantities? Let's start with one of the inputs:


In [19]:
MEq0 = Trace[np.where(Trace[:,1]==0)]
PFEq0GivenMEq1 = MEq0[np.where(MEq0[:,2]==0)].shape[0]/MEq0.shape[0]
print(f"P(M=0)={PFEq0GivenMEq1} - expected {PF_M[0,0]}")

P(M=0)=0.7995302532895615 - expected 0.8




Now try something we don't have: $P(M\vert F)$? We can do this exactly:

In [20]:
PM_F = (PF_M*PM).T/PF
print(PM_F)

[[0.85714286 0.5       ]
 [0.14285714 0.5       ]]


How do we get this from the trace? We need to
* Find the rows in the Trace corresponding to the desired condition
* Then compute the proportion of those rows with the desired outcome 

In [21]:
# Find the rows where F is True
FTrue = Trace[np.where(Trace[:,2]==1)]
# Add up column 1 and divide by the number of samples where F is True
MTrueFTrue = FTrue[:,1].sum()/FTrue.shape[0]
print(f"P(M=True|F=True) = {MTrueFTrue} ({PM_F[1][1]} expected)")

FFalse = Trace[np.where(Trace[:,2]==0)]
# Add up column 1 and divide by the number of samples where F is True
MTrueFFalse = FFalse[:,1].sum()/FFalse.shape[0]
print(f"P(M=True|F=False) = {MTrueFFalse} ({PM_F[1][0]} expected)")


P(M=True|F=True) = 0.5017578772802653 (0.4999999999999999 expected)
P(M=True|F=False) = 0.14227630637079455 (0.14285714285714285 expected)


This is an example of **rejection sampling**. In each case we have rejected all the rows in the Trace for which desired condition on $F$ was false. We may consider this to be a waste; however, if we want to ask general question of the distribution, the full trace in which all variables are drawn allows us to do so, as we demonstrated by computing by $P(M\vert F)$ and $P(M\vert \lnot F)$

Rejection sampling is very general and it can be shown that the standard deviation in the error in each probability is proportional to $1/\sqrt{N}$. However, if we have specific questions we want to answer and are time or compute-limited, then we may wish for something more efficient. This will be our next topic.